# Missing Child Finder - Free Colab GPU Engine
Given your Intel i5 setup, this notebook offloads the heavy AI Inference entirely to Google Colab's Free T4 GPU.

### Important Prep:
1. Go to **Runtime > Change runtime type** in the top menu.
2. Select **T4 GPU** as the Hardware Accelerator.
3. Run the cells below.

In [ ]:
!pip install fastapi uvicorn pydantic numpy opencv-python-headless insightface onnxruntime-gpu torch torchvision albumentations pycocotools pyngrok

In [ ]:
%%writefile server.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import base64, numpy as np, cv2, insightface
from insightface.app import FaceAnalysis
import uvicorn
from pyngrok import ngrok

app = FastAPI(title="MCF GPU Engine")

print("Loading RetinaFace & ArcFace Models into GPU...")
face_app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640, 640))

class AnalyzeRequest(BaseModel):
    frame_b64: str
    reference_embedding: list[float]

@app.get("/")
def health_check(): return {"status": "ok", "gpu": True}

@app.post("/analyze")
def analyze_frame(req: AnalyzeRequest):
    img_data = base64.b64decode(req.frame_b64)
    nparr = np.frombuffer(img_data, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    faces = face_app.get(img)
    ref_emb = np.array(req.reference_embedding, dtype=np.float32)
    matches = []

    for face in faces:
        sim = np.dot(face.embedding, ref_emb) / (np.linalg.norm(face.embedding) * np.linalg.norm(ref_emb))
        if sim > 0.68:
            matches.append({"confidence": float(sim), "bbox": face.bbox.tolist()})

    return {"matches": matches}

# ---> YOUR NGROK TOKEN AUTOMATICALLY INJECTED BELOW!
ngrok_token = "3C6v4S7HhMVKSUUMc4135dPsIUD_3U8iXKHMAS6mudmortCaZ"
ngrok.set_auth_token(ngrok_token)
public_url = ngrok.connect(8050).public_url

print("\n" + "="*50)
print(f"⚡ YOUR ML_SERVICE_URL IS: {public_url} ⚡")
print("COPY THIS URL AND PASTE IT IN YOUR LOCAL TERMINAL!")
print("Example: $env:ML_SERVICE_URL=\"" + public_url + "\"")
print("="*50 + "\n")

if __name__ == "__main__":
    uvicorn.run("server:app", host="0.0.0.0", port=8050)

In [ ]:
!python server.py